# 07 - SharePoint: Content Understanding, embeddings and Entra ACLs

## Required permissions

### User permissions

| Who needs it | Task | Required permission or role | Scope / when needed |
|---|---|---|---|
| User running the Search setup cells | Create indexes, data sources, skillsets and indexers | **Search Service Contributor** | Azure AI Search service |
| Each signed-in user running queries | Query indexed documents | **Search Index Data Reader** (Data Contributor also works), plus access to the source documents | Search service and SharePoint documents |
| User configuring the app registration | Create/configure the ingestion app and its federated credential | Permission to register apps and manage this app, such as app ownership or **Application Administrator** | Microsoft Entra; one-time setup |
| Administrator granting tenant consent | Grant the app's API permissions admin consent | **Privileged Role Administrator** or **Global Administrator** | Microsoft Entra; includes Microsoft Graph application permissions |
| Administrator granting site access | Grant the app access to the test site using the delegated setup path | **SharePoint Administrator** or higher, site membership, and Graph delegated **`Sites.FullControl.All`** in the calling token | Target SharePoint site; one-time setup |

The final two tasks can be completed by separately authorized accounts. Azure subscription Owner does not replace Entra or SharePoint authorization. Once app setup is complete, running ingestion and queries needs only the first two rows. Section 2c also documents an explicit app-only alternative for the site grant.

### App registration permissions

Configure these on the **ingestion app**, not on the notebook user's account. All API permissions below are **Application** permissions and require admin consent.

| Who needs it | API / resource | Permission | Purpose |
|---|---|---|---|
| Ingestion app's service principal | Microsoft Graph | **`Files.Read.All`** | Read document-library files for ingestion; tenant-wide file read |
| Ingestion app's service principal | Microsoft Graph | **`Sites.Selected`** | Selected-site authorization for document ACL ingestion |
| Ingestion app's service principal | Target SharePoint site | Explicit **Read** grant to the app | Authorize the selected site separately from admin consent |

The app also needs a **federated credential trusting Search's system-assigned managed identity**; no client secret is used. `Sites.Selected` does not narrow the separate tenant-wide `Files.Read.All` grant. The setup user's `Sites.FullControl.All` is not a permanent runtime permission for this app.

| Who needs it | Permission | Scope / purpose |
|---|---|---|
| Search system-assigned managed identity | **Cognitive Services User** | Parent Foundry resource; keyless Content Understanding and text embeddings. Bicep already declares this grant. |

Sources: [Search ACL permission matrix](https://learn.microsoft.com/azure/search/search-indexer-sharepoint-access-control-lists#permissions-by-acl-scenario), [admin consent roles](https://learn.microsoft.com/entra/identity/enterprise-apps/grant-admin-consent#prerequisites), [site-grant permissions](https://learn.microsoft.com/graph/api/site-post-permissions#permissions).

**Resource prefix:** use `WORKSHOP_PREFIX`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). The index, data source, skillset and indexer use `alice01-nb7-*`, following the reference notebook's naming. Existing `alice01-sp-*` objects are left unchanged. Optional app creation uses `alice01-sharepoint-indexer`; normal runs reuse the configured app. Prefixes prevent naming collisions, not enforce access isolation.

This is the text-only adaptation of [01_setup_index.ipynb](https://github.com/memasanz/foundry-iq-sharepoint-indexer/blob/main/notebooks/01_setup_index.ipynb): a small REST helper, Content Understanding semantic chunks, text embeddings, page/source metadata and ACL-preserving projections. There are no Vision skills, image vectorizers, normalized-image projections or image-description model calls. The same content can be queried without an index in **notebook 08 (remote)**.

| What runs | Where content lives | App registration |
|---|---|---|
| SharePoint indexer -> Content Understanding -> text embeddings -> ACL-preserving projections -> Search index | Markdown chunks, text vectors and source/page metadata in Azure AI Search | Required for ingestion |

This lab demonstrates **Entra user and group ACLs**, not native SharePoint-group expansion. Give test users or Entra groups access directly to the test documents. Changing the extraction pipeline does not fix unsupported group nesting or make every document visible in SharePoint visible through Search; verify allowed and restricted users separately.

**Run order:** run section 1, complete the one-time app setup in section 2, then run sections 3-6. `scripts\setup-sharepoint-app.ps1` automates app configuration and the site grant. Reuse an already configured app rather than creating another. Cleanup is opt-in.

## Prerequisites and safe test content

- Use the workshop tenant and a dedicated **non-sensitive test SharePoint site**, not a production library. Search and SharePoint should be in the same tenant for this lab. Sign in with `az login --tenant <tenant-id>`.
- Complete the [SharePoint indexer preview registration](https://aka.ms/azure-cognitive-search/indexer-preview). Review current preview limitations, including Conditional Access incompatibilities, unsupported guest-user ACLs and unsupported Entra groups nested inside SharePoint groups. Do not disable tenant security policies to make a lab run.
- Use the user and app permission tables above. Complete admin consent, the selected-site grant and managed-identity federation before running ingestion. The optional app-creation and trust-check cells additionally need Graph app-management/read authorization in the calling token; leave them disabled when reusing an already configured app.
- Prepare two small, non-sensitive documents in the test site's default Documents library. Give one test user or an Entra group direct access to the restricted document; the second test user must not have another access path. Do not rely on an Entra group nested inside a SharePoint group, or on site/group ownership alone.
- Reuse the deployed Foundry resource and `workshop-embedding` deployment. Content Understanding needs a supported Foundry region, normally East US, and the Search identity's Cognitive Services User role. Analysis and embeddings are billable; analysis can time out after five minutes and still incur charges. This text-only pipeline needs no chat-model deployment.
- Add `SHAREPOINT_SITE_URL` and a unique `WORKSHOP_PREFIX` to your untracked `.env`. Set the registration identifiers after section 2. Never paste access tokens or client secrets into cells, output, or chat.

These are preview APIs. Review the linked documentation before running.

In [ ]:
import json
import os
import re
import time
from pathlib import Path
from urllib.parse import urlparse
from uuid import UUID

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env', override = True)
load_dotenv(root / '.env.azure', override = True)
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'SHAREPOINT_SITE_URL',
            'AZURE_AI_ACCOUNT_NAME', 'AZURE_AI_EMBEDDING_DEPLOYMENT_NAME')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. See .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
prefix = os.getenv('WORKSHOP_PREFIX') or input('Resource prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')
site_url = os.environ['SHAREPOINT_SITE_URL'].rstrip('/')
site = urlparse(site_url)
if (site.scheme != 'https' or not site.hostname or not site.hostname.endswith('.sharepoint.com')
        or site.username or site.password or site.port not in (None, 443)
        or site.query or site.fragment or any(c in site_url for c in ';"\r\n')):
    raise ValueError('Use a public-cloud SharePoint HTTPS site URL, not a file or sharing-link URL.')
search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
api_version = '2026-08-01-preview'
index_name = f'{prefix}-nb7-index'
data_source_name = f'{prefix}-nb7-ds'
skillset_name = f'{prefix}-nb7-skillset'
indexer_name = f'{prefix}-nb7-indexer'
foundry_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.services.ai.azure.com"
embedding_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.openai.azure.com"
embedding_deployment = os.environ['AZURE_AI_EMBEDDING_DEPLOYMENT_NAME']
embedding_model = os.getenv('AZURE_AI_EMBEDDING_MODEL_NAME', 'text-embedding-3-small')
embedding_dimensions = 1536
semantic_configuration = 'semantic-config'
credential = AzureCliCredential(tenant_id=tenant_id)
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
graph_token = get_bearer_token_provider(credential, 'https://graph.microsoft.com/.default')
arm_token = get_bearer_token_provider(credential, 'https://management.azure.com/.default')
print('Search:', search_endpoint)
print('SharePoint site:', site_url)
print('API version:', api_version)
print('Index:', index_name, '| Foundry enrichment:', foundry_endpoint)


In [ ]:
def search_request(method, path, body=None, extra_headers=None, allow_not_found=False):
    headers = {'Authorization': f'Bearer {search_token()}'}
    if extra_headers:
        headers.update(extra_headers)
    response = requests.request(
        method, f'{search_endpoint}/{path}?api-version={api_version}',
        headers=headers, json=body, timeout=(10, 90),
    )
    if allow_not_found and response.status_code == 404:
        return None
    if response.status_code >= 400:
        print(f'{method} {path}: {response.status_code}', str(response.text)[:2000])
    response.raise_for_status()
    return {} if response.status_code in (202, 204) else response.json()


## 2. Register and authorize the SharePoint ingestion app

`scripts\setup-sharepoint-app.ps1` performs 2a, 2b, the site grant in 2c when `-SiteUrl` is provided, and optional consent in one run; the steps below explain each operation. Complete setup before creating the SharePoint data source. One configured ingestion app can be reused by multiple indexes on this Search service.

### 2a. Create the application and understand its IDs

In **Microsoft Entra ID -> App registrations -> New registration**, create `iq-workshop-sharepoint-indexer` as **single tenant**. The secretless ingestion flow needs no redirect URI, no public-client flow and no client secret. Record the **Application (client) ID** and the app registration's **Object ID** in `.env` as `SHAREPOINT_APP_CLIENT_ID` and `SHAREPOINT_APP_OBJECT_ID`.

Alternatively, the following **opt-in** cell creates a new application and service principal using Microsoft Graph. It declares permissions but does **not** grant admin consent, create a federated credential or grant site access. Your signed-in CLI identity must have the Graph authorization required to create apps/service principals; a 403 is not fixed by giving yourself a subscription role. Use the portal instead if your tenant does not permit this CLI workflow. Do not create a second app if the portal app already exists.

Use the **App registration permissions** table at the start of this notebook. This Entra ACL path needs the two Microsoft Graph application permissions, not SharePoint API permissions for native-group resolution. The shared setup script also configures native-group permissions for other scenarios; an existing app with that superset can be reused without changing its grants.

**Scope warning:** `Sites.Selected` requires an explicit grant on each workshop site; consent alone is insufficient. However, the separately required **Graph `Files.Read.All` is tenant-wide file-read permission**. This documented combination is **not strict site-only isolation** of the app. Review this with the tenant administrator and use a dedicated test tenant/site. We do not request tenant-wide `Sites.FullControl.All`. Do not silently remove a required permission or claim that the site URL in the data source limits the app's underlying authorization.

In the portal, add these permissions under their **correct API surfaces**, select **Application permissions**, then have an authorized administrator choose **Grant admin consent**. Requested permissions and consented permissions are different states.

In [ ]:
CREATE_APP_REGISTRATION = False
if CREATE_APP_REGISTRATION:
    if os.getenv('SHAREPOINT_APP_CLIENT_ID'):
        raise RuntimeError('An app is already configured. Reuse it; do not create a duplicate.')
    required_access = []
    for resource_app_id, permission_names in [
        ('00000003-0000-0000-c000-000000000000', ['Files.Read.All', 'Sites.Selected']),
    ]:
        response = requests.get(
            'https://graph.microsoft.com/v1.0/servicePrincipals',
            headers={'Authorization': f'Bearer {graph_token()}'},
            params={'$filter': f"appId eq '{resource_app_id}'", '$select': 'id,appId,appRoles'},
            timeout=(10, 90),
        )
        response.raise_for_status()
        resources = response.json()['value']
        if len(resources) != 1:
            raise RuntimeError(f'Expected one API service principal for {resource_app_id}.')
        roles = {role['value']: role['id'] for role in resources[0]['appRoles']
                 if role.get('isEnabled') and 'Application' in role['allowedMemberTypes']}
        required_access.append({
            'resourceAppId': resource_app_id,
            'resourceAccess': [{'id': roles[name], 'type': 'Role'} for name in permission_names],
        })
    response = requests.post(
        'https://graph.microsoft.com/v1.0/applications',
        headers={'Authorization': f'Bearer {graph_token()}'},
        json={'displayName': f'{prefix}-sharepoint-indexer', 'signInAudience': 'AzureADMyOrg',
              'requiredResourceAccess': required_access}, timeout=(10, 90),
    )
    response.raise_for_status()
    app = response.json()
    os.environ['SHAREPOINT_APP_CLIENT_ID'] = app['appId']
    os.environ['SHAREPOINT_APP_OBJECT_ID'] = app['id']
    print('Record in .env before continuing:')
    print('SHAREPOINT_APP_CLIENT_ID=' + app['appId'])
    print('SHAREPOINT_APP_OBJECT_ID=' + app['id'])
    response = requests.post(
        'https://graph.microsoft.com/v1.0/servicePrincipals',
        headers={'Authorization': f'Bearer {graph_token()}'},
        json={'appId': app['appId']}, timeout=(10, 90),
    )
    response.raise_for_status()
    print('Service principal created. Admin consent and site authorization are still required.')
else:
    print('No app created. Reuse the configured app, use the portal walkthrough, or explicitly enable app creation.')


### 2b. Trust Search's managed identity (no client secret)

1. Open the Search resource -> **Identity** -> **System assigned**. It is enabled by Bicep. Note its **Object (principal) ID**.
2. Open **Entra ID -> Enterprise applications**, find this managed identity by its principal ID and copy its **Application ID**. Save that as `SHAREPOINT_SEARCH_MI_CLIENT_ID` in `.env`.
3. On the **ingestion app registration**, open **Certificates & secrets -> Federated credentials -> Add credential**. Choose the **Managed Identity** scenario, select this Search system-assigned identity and save a named credential. Do not create a client secret.
4. Verify the trust points at the exact Search identity, not the original/deleted Foundry account, your user or the Foundry project's identity.

| Identifier | Used for |
|---|---|
| Ingestion app client ID | `ApplicationId` in the data source connection string |
| Search managed identity client/application ID | `FederatedCredentialApplicationId` in the data source connection string |
| Search managed identity principal/object ID | Subject trusted by the app's federated credential |
| Ingestion app object ID | Graph `/applications/{id}` operations only |
| Federated credential object's own ID | Not used in the data source connection string |

### 2c. Grant the app access to the selected test site

An administrator must make an explicit site permission grant **in addition to Entra admin consent**. The setup script grants site-scoped **Read** on the dedicated test site. The Search ACL permission matrix documents `Sites.Selected` and an explicit site grant, but does not specify the minimum site-grant role for every ACL scenario. Verify ACL extraction and two-user query behavior; content-ingestion success alone does not prove ACL support. Do not automatically escalate to FullControl or substitute a production site.

Run the setup script in a **separate PowerShell terminal** from the repository root, using the ingestion app's display name and your site URL:

```powershell
.\scripts\setup-sharepoint-app.ps1 -AppName 'iq-workshop-sharepoint-indexer' `
  -SiteUrl 'https://<tenant>.sharepoint.com/sites/<workshop-site>'
```

The script grants and verifies site-scoped Read directly through Microsoft Graph. Existing grants that allow reading are reused; broader grants are left unchanged with a warning. No additional app registration or PnP setup is needed. By default, the signed-in caller needs Graph delegated `Sites.FullControl.All`, a SharePoint Administrator (or higher) role, and target-site membership; unauthorized preflight stops before changing the app.

If the CLI token cannot manage site grants, an authorized **Privileged Role Administrator or Global Administrator** can explicitly opt in to app-only bootstrap instead:

```powershell
.\scripts\setup-sharepoint-app.ps1 -AppName 'iq-workshop-sharepoint-indexer' `
  -SiteUrl 'https://<tenant>.sharepoint.com/sites/<workshop-site>' `
  -BootstrapSiteAccess -GrantAdminConsent
```

Bootstrap requires Graph authorization to manage credentials and app-role assignments. It temporarily consents tenant-wide Graph `Sites.FullControl.All` on the ingestion app's service principal and adds a 15-minute credential to grant the site **Read**, then attempts and verifies cleanup in `finally`, including on failure. No runtime secret is added, and pre-existing credentials/permissions are preserved. Tokens/credentials are not printed or written to `.env`; issued tokens can remain valid until expiry. Forced termination can prevent cleanup; any reported cleanup failure requires manual action. This option does not bypass administrator authorization. Runtime admin consent remains separate/opt-in (`-GrantAdminConsent`).

### 2d. Optional read-only trust check

The following optional cell validates the managed-identity identifier and federated trust via ARM and Graph. It requires read access to the Search identity and the app's federated credentials. It does not grant consent or site access; check those separately in the portal and the site-grant output.

In [ ]:
VERIFY_FEDERATION = False
if VERIFY_FEDERATION:
    app_object_id = str(UUID(os.environ['SHAREPOINT_APP_OBJECT_ID']))
    expected_mi_client_id = str(UUID(os.environ['SHAREPOINT_SEARCH_MI_CLIENT_ID']))
    search_id = os.environ['AZURE_SEARCH_RESOURCE_ID']
    response = requests.get(
        f'https://management.azure.com{search_id}?api-version=2025-05-01',
        headers={'Authorization': f'Bearer {arm_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    mi_principal_id = response.json()['identity']['principalId']
    response = requests.get(
        f'https://graph.microsoft.com/v1.0/servicePrincipals/{mi_principal_id}',
        headers={'Authorization': f'Bearer {graph_token()}'},
        params={'$select': 'id,appId'}, timeout=(10, 90),
    )
    response.raise_for_status()
    if str(UUID(response.json()['appId'])) != expected_mi_client_id:
        raise RuntimeError('SHAREPOINT_SEARCH_MI_CLIENT_ID is not this Search managed identity.')
    response = requests.get(
        f'https://graph.microsoft.com/v1.0/applications/{app_object_id}',
        headers={'Authorization': f'Bearer {graph_token()}'},
        params={'$select': 'id,appId'}, timeout=(10, 90),
    )
    response.raise_for_status()
    if str(UUID(response.json()['appId'])) != str(UUID(os.environ['SHAREPOINT_APP_CLIENT_ID'])):
        raise RuntimeError('App object ID and ingestion app client ID do not match.')
    response = requests.get(
        f'https://graph.microsoft.com/v1.0/applications/{app_object_id}/federatedIdentityCredentials',
        headers={'Authorization': f'Bearer {graph_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    trusts = response.json()['value']
    if not any(trust.get('subject') == mi_principal_id
               and trust.get('issuer') == f'https://login.microsoftonline.com/{tenant_id}/v2.0'
               and 'api://AzureADTokenExchange' in trust.get('audiences', []) for trust in trusts):
        raise RuntimeError('No matching managed-identity federated trust exists on this app.')
    print('Identity IDs and trust match. Still verify admin consent and the selected-site grant.')
else:
    print('Read-only trust check skipped; verify the federated credential in the portal before ingestion.')


## 3. Indexed path: create the SharePoint data source

After completing the registration, consent, federation and site grant, set `INDEXED_SETUP_CONFIRMED=True`. This confirms that setup was reviewed; it is **not an automated permissions test**. A successful indexer run and two-user permission exercise are still required.

The connection string contains only non-secret IDs. Reuse the existing app and federation rather than introducing a client secret. This authenticates ingestion; the query token later identifies the user whose document permissions Search evaluates.

`defaultSiteLibrary` scopes this lab to the selected site's default document library. It does not crawl all tenant sites, lists, or site pages. All files in that test library can be indexed; use a dedicated library/site containing only workshop content.

In [ ]:
INDEXED_SETUP_CONFIRMED = False
if not INDEXED_SETUP_CONFIRMED:
    raise RuntimeError('Complete section 2, then explicitly acknowledge it before indexed ingestion.')
sp_app_client_id = str(UUID(os.environ['SHAREPOINT_APP_CLIENT_ID']))
sp_mi_client_id = str(UUID(os.environ['SHAREPOINT_SEARCH_MI_CLIENT_ID']))
if sp_app_client_id == sp_mi_client_id:
    raise ValueError('The ingestion app and Search managed identity must have different client IDs.')
connection_string = (
    f'SharePointOnlineEndpoint={site_url};ApplicationId={sp_app_client_id};'
    f'FederatedCredentialApplicationId={sp_mi_client_id};TenantId={tenant_id}'
)
print('Indexed path configured. No client secret is used.')


In [ ]:
data_source = {
    'name': data_source_name, 'type': 'sharepoint',
    'credentials': {'connectionString': connection_string},
    'container': {'name': 'defaultSiteLibrary'},
    'indexerPermissionOptions': ['userIds', 'groupIds'],
}
search_request('PUT', f'datasources/{data_source_name}', data_source)
print('SharePoint data source ready:', data_source_name)


### Index schema: text vectors, source metadata and Entra ACLs

`permissionFilterOption='enabled'` makes Search match the querying user's Entra identity against `UserIds` and `GroupIds`. The ACL fields are filterable but not retrievable in normal results. Like the reference notebook, this index does not configure native SharePoint-group resolution.

`content` holds Markdown, `contentVector` holds its embedding, and `sourceFile`, `webUrl`, `page` and `pageTo` identify the evidence. SharePoint ACL ingestion also requires the exact `metadata_spo_item_path` field; a friendly web URL does not replace it. Project the source path and both ACL collections onto every chunk.

The embedding skill and query-time Azure OpenAI vectorizer use the same deployment, model and 1536 dimensions. The initial wildcard query deliberately checks visibility before testing relevance.

In [ ]:
index = {
    'name': index_name,
    'permissionFilterOption': 'enabled',
    'fields': [
        {'name': 'id', 'type': 'Edm.String', 'key': True,
         'searchable': True, 'analyzer': 'keyword'},
        {'name': 'parent_id', 'type': 'Edm.String', 'filterable': True},
        {'name': 'content', 'type': 'Edm.String', 'searchable': True},
        {'name': 'contentVector', 'type': 'Collection(Edm.Single)',
         'searchable': True, 'retrievable': False, 'dimensions': embedding_dimensions,
         'vectorSearchProfile': 'text-profile'},
        {'name': 'sourceFile', 'type': 'Edm.String', 'searchable': True, 'filterable': True},
        {'name': 'webUrl', 'type': 'Edm.String'},
        {'name': 'page', 'type': 'Edm.Int32', 'filterable': True},
        {'name': 'pageTo', 'type': 'Edm.Int32', 'filterable': True},
        {'name': 'metadata_spo_item_id', 'type': 'Edm.String', 'filterable': True},
        {'name': 'metadata_spo_item_path', 'type': 'Edm.String',
         'searchable': False, 'filterable': True, 'retrievable': False},
        {'name': 'lastModified', 'type': 'Edm.DateTimeOffset', 'filterable': True},
        {'name': 'UserIds', 'type': 'Collection(Edm.String)', 'permissionFilter': 'userIds',
         'filterable': True, 'retrievable': False},
        {'name': 'GroupIds', 'type': 'Collection(Edm.String)', 'permissionFilter': 'groupIds',
         'filterable': True, 'retrievable': False},
    ],
    'vectorSearch': {
        'algorithms': [{'name': 'hnsw', 'kind': 'hnsw',
                        'hnswParameters': {'metric': 'cosine'}}],
        'profiles': [{'name': 'text-profile', 'algorithm': 'hnsw', 'vectorizer': 'aoai-vectorizer'}],
        'vectorizers': [{
            'name': 'aoai-vectorizer', 'kind': 'azureOpenAI',
            'azureOpenAIParameters': {
                'resourceUri': embedding_endpoint, 'deploymentId': embedding_deployment,
                'modelName': embedding_model,
            },
        }],
    },
    'semantic': {
        'defaultConfiguration': semantic_configuration,
        'configurations': [{
            'name': semantic_configuration,
            'prioritizedFields': {
                'titleField': {'fieldName': 'sourceFile'},
                'prioritizedContentFields': [{'fieldName': 'content'}],
            },
        }],
    },
}
search_request('PUT', f'indexes/{index_name}', index)
print('ACL-aware index ready:', index_name)


## 4. Content Understanding, text embeddings and ACL projections

Content Understanding reads the original file and emits Markdown `text_sections` using the reference notebook's semantic chunking: up to 800 tokens per section, with no overlap. `locationMetadata` supplies page ranges. Each section is embedded with the workshop text-embedding deployment; no Text Split or image-processing skill is used.

**Every chunk must inherit the source ACLs.** The projection copies `/document/metadata_user_ids` and `/document/metadata_group_ids` into the permission-filter fields. With `skipIndexingParentDocuments`, parent-only indexer field mappings are insufficient. We avoid knowledge stores, enrichment caches, debug sessions and custom Web API skills, which have SharePoint ACL preview limitations.

In [ ]:
skillset = {
    'name': skillset_name,
    'description': 'Content Understanding semantic text sections, embeddings and Entra ACLs.',
    'cognitiveServices': {
        '@odata.type': '#Microsoft.Azure.Search.AIServicesByIdentity',
        'subdomainUrl': foundry_endpoint,
    },
    'skills': [
        {
            '@odata.type': '#Microsoft.Skills.Util.ContentUnderstandingSkill',
            'name': 'content-understanding', 'context': '/document',
            'extractionOptions': ['locationMetadata'],
            'chunkingProperties': {'method': 'semantic', 'unit': 'tokens',
                                   'maximumLength': 800, 'overlapLength': 0},
            'inputs': [{'name': 'file_data', 'source': '/document/file_data'}],
            'outputs': [{'name': 'text_sections', 'targetName': 'text_sections'}],
        },
        {
            '@odata.type': '#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill',
            'name': 'text-embed', 'context': '/document/text_sections/*',
            'resourceUri': embedding_endpoint, 'deploymentId': embedding_deployment,
            'modelName': embedding_model, 'dimensions': embedding_dimensions,
            'inputs': [{'name': 'text', 'source': '/document/text_sections/*/content'}],
            'outputs': [{'name': 'embedding', 'targetName': 'content_vector'}],
        },
    ],
    'indexProjections': {
        'selectors': [{
            'targetIndexName': index_name, 'parentKeyFieldName': 'parent_id',
            'sourceContext': '/document/text_sections/*',
            'mappings': [
                {'name': 'content', 'source': '/document/text_sections/*/content'},
                {'name': 'contentVector', 'source': '/document/text_sections/*/content_vector'},
                {'name': 'page', 'source': '/document/text_sections/*/locationMetadata/pageNumberFrom'},
                {'name': 'pageTo', 'source': '/document/text_sections/*/locationMetadata/pageNumberTo'},
                {'name': 'sourceFile', 'source': '/document/metadata_spo_item_name'},
                {'name': 'metadata_spo_item_id', 'source': '/document/metadata_spo_item_id'},
                {'name': 'webUrl', 'source': '/document/metadata_spo_item_weburi'},
                {'name': 'metadata_spo_item_path', 'source': '/document/metadata_spo_item_path'},
                {'name': 'lastModified', 'source': '/document/metadata_spo_item_last_modified'},
                {'name': 'UserIds', 'source': '/document/metadata_user_ids'},
                {'name': 'GroupIds', 'source': '/document/metadata_group_ids'},
            ],
        }],
        'parameters': {'projectionMode': 'skipIndexingParentDocuments'},
    },
}
search_request('PUT', f'skillsets/{skillset_name}', skillset)
print('ACL-preserving skillset ready:', skillset_name)


## 5. Create and monitor the SharePoint indexer

The first run starts on creation. Updating an existing idle indexer in this cell explicitly requests a new run; wait if it is already running. The SharePoint preview registration, application permissions, admin consent, selected-site grant, federated credential and Search permissions must all be correct. A successful data-source PUT alone does not prove that ingestion can read the site. No schedule is set; reruns are explicit.

In [ ]:
status = search_request('GET', f'indexers/{indexer_name}/status', allow_not_found=True)
indexer_exists = status is not None
previous_run_start = None
if indexer_exists:
    previous_run = status.get('lastResult') or {}
    if previous_run.get('status') == 'inProgress':
        raise RuntimeError('The indexer is already running. Wait before updating it.')
    previous_run_start = previous_run.get('startTime')
indexer = {
    'name': indexer_name, 'dataSourceName': data_source_name,
    'targetIndexName': index_name, 'skillsetName': skillset_name,
    'parameters': {
        'batchSize': 1,
        'maxFailedItems': 0, 'maxFailedItemsPerBatch': 0,
        'configuration': {
            'dataToExtract': 'contentAndMetadata', 'allowSkillsetToReadFileData': True,
        },
    },
}
search_request('PUT', f'indexers/{indexer_name}', indexer)
print('SharePoint indexer ready:', indexer_name)
if indexer_exists:
    search_request('POST', f'indexers/{indexer_name}/run')
    print('New run requested for the existing indexer.')


In [ ]:
for attempt in range(90):
    status = search_request('GET', f'indexers/{indexer_name}/status')
    last_run = status.get('lastResult') or {}
    state = last_run.get('status')
    print('Run:', state, '| processed:', last_run.get('itemsProcessed'),
          '| failed:', last_run.get('itemsFailed'))
    is_requested_run = previous_run_start is None or last_run.get('startTime') != previous_run_start
    if state == 'success' and is_requested_run:
        print('Warnings:', json.dumps(last_run.get('warnings', []), indent=2))
        if last_run.get('itemsFailed', 0) or last_run.get('errors'):
            raise RuntimeError(json.dumps(last_run, indent=2))
        break
    if (is_requested_run and state in ('transientFailure', 'persistentFailure', 'error')) or status.get('status') == 'error':
        raise RuntimeError(json.dumps(last_run or status, indent=2))
    if attempt < 89:
        time.sleep(10)
else:
    raise TimeoutError('Indexer did not complete within 15 minutes. Inspect its status before retrying.')


## 6. Query indexed content as the signed-in user

`Authorization` authenticates access to Search. `x-ms-query-source-authorization` supplies the end-user identity for document permission filtering. In this local lab both represent your CLI-signed-in user, but their purposes are different. Do not substitute an application-only token for the end-user token.

Start with `question = '*'` to check document visibility independently of keyword matching. Then replace it with terms from your documents. The sample travel-policy query will not necessarily match an existing library's content.

`UserIds` contains Entra user object IDs; `GroupIds` contains the source's group permission entries. For this notebook, use Entra groups assigned directly to documents. Native SharePoint `spg:` entries and special owners claims are not ordinary Entra group GUIDs; this index does not expand them. A document can be visible in SharePoint through a different access path and still be filtered out here. See [supported group relationships](https://learn.microsoft.com/azure/search/search-indexer-sharepoint-access-control-lists#supported-group-relationships).

There is **no elevated-read header** in this lab. Search Index Data Contributor can perform elevated reads; do not use that privilege to claim that an ordinary user can access a document. For production query clients, use appropriately restricted Search roles.

An empty result is valid when no authorized content matches. It is not proof that the indexer failed or that all permissions are correct.

In [ ]:
question = '*'
indexed_result = search_request(
    'POST', f'indexes/{index_name}/docs/search',
    {'search': question, 'count': True, 'top': 5,
     'select': 'sourceFile,webUrl,page,pageTo,content'},
    extra_headers={'x-ms-query-source-authorization': search_token()},
)
print('Visible chunks:', indexed_result.get('@odata.count'))
print(json.dumps(indexed_result, indent=2))


### Prove permission trimming and refresh behavior

Use two **non-admin member accounts**, not guest users, and confirm their actual access by opening the restricted document in SharePoint. Give account A access directly, then repeat using an **Entra group assigned directly to the document**. Account B must not have access via another group, sharing link or site-admin privilege. Run the same query and index name for both accounts, signing out/in to Azure CLI and restarting the kernel between identities. Both need Search Index Data Reader; creating Search objects separately requires Search Service Contributor.

Expected: account A can retrieve the restricted document; account B cannot. Remove the direct user grant before the group-only check so that a `UserIds` match cannot hide a group-resolution failure. Record document URLs and observed results without saving source content into the committed notebook. The notebook never changes SharePoint permissions automatically, and a successful direct-user test does not prove group access.

Next change document permissions, then opt in to run the indexer again. Item-level unique ACL changes refresh on successful runs. **Inherited site/library/folder ACL changes need an explicit permissions resync** (or a supported document reset) and then an indexer run. The following cell demonstrates the resync; it touches only this named indexer. Rerun the monitor and query cells afterwards.

Changing a skillset instead requires reprocessing content, not just an ACL resync. Indexed content/permissions have synchronization latency; remote retrieval follows Microsoft 365's own indexing and permission-propagation behavior, not instantaneous document scanning.

In [ ]:
RUN_AGAIN = False
RESYNC_PERMISSIONS = False
if RUN_AGAIN:
    status = search_request('GET', f'indexers/{indexer_name}/status')
    last_run = status.get('lastResult') or {}
    if last_run.get('status') == 'inProgress':
        raise RuntimeError('The indexer is already running. Wait for it to finish.')
    previous_run_start = last_run.get('startTime')
    if RESYNC_PERMISSIONS:
        search_request('POST', f'indexers/{indexer_name}/resync', {'options': ['permissions']})
    search_request('POST', f'indexers/{indexer_name}/run')
    print('Run requested. Rerun monitor-indexer, then query-index.')
else:
    print('No additional indexer run or ACL resync requested.')


## 7. Optional Search-object cleanup and separate tenant cleanup

Enable the following cell to remove only this notebook's named Search objects. Missing objects (404) are explicitly allowed; authorization failures and other errors are not ignored.

**This does not remove the ingestion app, its enterprise application, federated credential, consent, selected-site grant or source files.** First check whether other indexes still use the app. When no longer needed, use an authorized SharePoint/Graph administration workflow to revoke the site grant and remove the federated trust/consented permissions or delete the dedicated test app and service principal. Remove only documents/site created for this exercise, not a pre-existing app or site. App registrations are tenant objects and are not deleted by deleting the Azure resource group.

Clear all notebook outputs before sharing; outputs can contain private SharePoint document text and URLs even though tokens are not printed.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    for collection, name in [
        ('indexers', indexer_name), ('skillsets', skillset_name),
        ('indexes', index_name), ('datasources', data_source_name),
    ]:
        search_request('DELETE', f'{collection}/{name}', allow_not_found=True)
        print('Deleted or already absent:', name)
else:
    print('Search-object cleanup disabled. Tenant app/site cleanup is always a separate admin step.')


## References and sample lineage

Adapted from [foundry-iq-sharepoint-indexer: 01_setup_index.ipynb](https://github.com/memasanz/foundry-iq-sharepoint-indexer/blob/main/notebooks/01_setup_index.ipynb). The text extraction, embeddings, source/page metadata, Entra ACL projections and REST query pattern are retained. The Vision vectorizer/skill, image projections and image descriptions are omitted. Workshop environment names and explicit CLI-user authentication are retained so the query identity is unambiguous.

- [SharePoint indexer and app registration](https://learn.microsoft.com/azure/search/search-how-to-index-sharepoint-online)
- [ACL permissions matrix, native SharePoint groups, identifier lookup and refresh](https://learn.microsoft.com/azure/search/search-indexer-sharepoint-access-control-lists)
- [Grant a site permission with Microsoft Graph](https://learn.microsoft.com/graph/api/site-post-permissions)
- [Query-time ACL enforcement](https://learn.microsoft.com/azure/search/search-query-access-control-rbac-enforcement)
- [Content Understanding skill and semantic chunking](https://learn.microsoft.com/azure/search/cognitive-search-skill-content-understanding)

Skill/API versions and preview access can change. This notebook does not promise complete SharePoint permission-model parity. For retrieval without an index, continue with notebook 08.